# Woche 06: Strings, Reads und exaktes Matching mit Python

**Block B · Woche 06 · Selbststudium · Starter (Skeleton)**

Dieses Notebook enthält die Python-Beispiele aus dem Skriptum *„Strings, Reads und das Matching-Problem“* (Woche 06)
als Programmieraufgaben. In Woche 6 gibt es noch keine Pflichtübung; das Notebook dient zum Üben und ist in etwa
100 Minuten lösbar.

## Lernziele
- Präfixe, Suffixe und das **reverse Komplement** mit Slicing berechnen (Skriptum 4)
- Reads als **zufällige Teilstrings** eines Genoms simulieren (Skriptum 2.2)
- Den **naiven Algorithmus** für exaktes Matching programmieren und seine Vergleiche zählen (Skriptum 5)
- **FASTA** und **FASTQ** einlesen (Skriptum 6.2)
- Echte Reads gegen das Genom von **phiX174** suchen und die Ergebnisse erklären (Skriptum 6.3 und 6.4)

## Zu implementierende Funktionen
| Teil | Funktion(en) | Skriptum |
|---|---|---|
| 1 | `prefixes`, `suffixes`, `reverse_complement` | 4.2, 4.3 |
| 2 | `random_dna`, `generate_reads` | 2.2 |
| 3 | `naive`, `find_all`, `naive_with_counts`, `comparison_bounds`, `estimate_runtime_years` | 5 |
| 4 | `read_genome`, `read_fastq` | 6.2 |
| 5 | `count_matched`, `circularize` | 6.3, 6.4 |

> Im **Starter** werfen diese Funktionen `NotImplementedError`. Ersetze den Rumpf durch deine eigene Implementierung
> und führe danach die Test-Zelle (`%%ipytest`) des jeweiligen Teils aus, bis alle Tests grün sind. Die
> **Musterlösung** zeigt eine mögliche Lösung.

## 0. Setup
Führe diese Zelle zuerst aus. Fehlende Pakete werden einmalig installiert. Die Datendateien `phix.fa` und
`ERR266411_1.first1000.fastq` werden im aktuellen Ordner, im Ordner `daten`, in `../daten` und in `..` gesucht.

In [ ]:
# Benötigte Pakete (werden bei Bedarf einmalig installiert)
import importlib, subprocess, sys
for package in ["ipytest", "pytest"]:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])

import collections
import os
import random
import time

import pytest
import ipytest
ipytest.autoconfig()

DATA_DIRS = [".", "daten", os.path.join("..", "daten"), ".."]


def data_path(filename: str) -> str:
    """Return the path of a data file, searching the folders in DATA_DIRS."""
    for folder in DATA_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} not found in {DATA_DIRS}. Copy it next to this notebook.")


for f in ["phix.fa", "ERR266411_1.first1000.fastq"]:
    print(f"{f}: {data_path(f)}")

## 1. Strings: Präfix, Suffix, reverses Komplement (Skriptum 4)

Ein **Präfix** ist ein Teilstring, der am Anfang beginnt, ein **Suffix** einer, der am Ende endet (Langmead,
*String definitions*). In Python: `s[:k]` und `s[-k:]` bzw. `s[i:]`.

**Aufgabe 1.1:** Schreibe `prefixes(s)` und `suffixes(s)`.
- `prefixes("ACGT")` liefert alle nichtleeren Präfixe, vom kürzesten zum längsten: `["A", "AC", "ACG", "ACGT"]`
- `suffixes("ACGT")` liefert alle nichtleeren Suffixe, vom längsten zum kürzesten: `["ACGT", "CGT", "GT", "T"]`

**Aufgabe 1.2:** Schreibe `reverse_complement(s)`. Ersetze jede Base durch ihren Partner (A↔T, C↔G, N bleibt N) und
lies das Ergebnis rückwärts (Skriptum 4.3).

*Tipps:* List Comprehension mit `range`; ein Dictionary für die Partnerbasen; `reversed(s)` oder `s[::-1]`.

In [ ]:
def prefixes(s: str) -> list[str]:
    """All non-empty prefixes of s, shortest first."""
    # Tipp: s[:k] für k = 1 ... len(s)
    raise NotImplementedError("TODO: implement prefixes")


def suffixes(s: str) -> list[str]:
    """All non-empty suffixes of s, longest first."""
    # Tipp: s[i:] für i = 0 ... len(s) - 1
    raise NotImplementedError("TODO: implement suffixes")


def reverse_complement(s: str) -> str:
    """Reverse complement of a DNA string (A<->T, C<->G, N stays N)."""
    # Tipp: complement = {"A": "T", "C": "G", "G": "C", "T": "A", "N": "N"}
    raise NotImplementedError("TODO: implement reverse_complement")

In [ ]:
s = "AACCGGTT"
print(s[2:6], s[:6], s[4:], s[-4:])     # Skriptum 4.2: CCGG AACCGG GGTT GGTT
print(prefixes("ACGT"))
print(suffixes("ACGT"))
print(reverse_complement("ACCGT"))      # Skriptum 4.3: ACGGT

In [ ]:
%%ipytest

def test_prefixes():
    assert prefixes("ACGT") == ["A", "AC", "ACG", "ACGT"]
    assert prefixes("") == []


def test_suffixes():
    assert suffixes("ACGT") == ["ACGT", "CGT", "GT", "T"]
    assert len(suffixes("GATTACA")) == 7


def test_prefix_and_suffix_slicing():
    s = "GATTACA"
    assert prefixes(s)[2] == "GAT"      # Präfix der Länge 3
    assert suffixes(s)[-2] == "CA"      # Suffix der Länge 2


def test_reverse_complement():
    assert reverse_complement("ACCGT") == "ACGGT"
    assert reverse_complement("AAAACCCGGT") == "ACCGGGTTTT"
    assert reverse_complement("ACGTN") == "NACGT"


def test_reverse_complement_twice_is_identity():
    s = "GATTACAGATTACA"
    assert reverse_complement(reverse_complement(s)) == s

## 2. Reads als zufällige Teilstrings (Skriptum 2.2)

Langmead beschreibt Reads als zufällig gezogene Teilstrings des Genoms. Das simulieren wir.

**Aufgabe 2.1:** Schreibe `random_dna(length, seed=None)`, die einen zufälligen String aus `A`, `C`, `G`, `T` der
gewünschten Länge liefert. Verwende `rng = random.Random(seed)` und `rng.choice("ACGT")`, damit das Ergebnis mit
gleichem `seed` reproduzierbar ist.

**Aufgabe 2.2:** Schreibe `generate_reads(genome, n_reads, read_len, seed=None)`. Jeder Read beginnt an einer
zufälligen Position `start` mit `0 <= start <= len(genome) - read_len` und ist `genome[start:start + read_len]`.

*Tipp:* `rng.randint(a, b)` liefert eine ganze Zahl zwischen `a` und `b` **einschließlich** `b`.

In [ ]:
def random_dna(length: int, seed: int | None = None) -> str:
    """Random DNA string over A, C, G, T (reproducible with seed)."""
    # Tipp: rng = random.Random(seed); rng.choice("ACGT")
    raise NotImplementedError("TODO: implement random_dna")


def generate_reads(genome: str, n_reads: int, read_len: int, seed: int | None = None) -> list[str]:
    """Draw n_reads substrings of length read_len from random start positions."""
    # Tipp: start = rng.randint(0, len(genome) - read_len)
    raise NotImplementedError("TODO: implement generate_reads")

In [ ]:
toy_genome = random_dna(60, seed=6)
print(toy_genome)
for read in generate_reads(toy_genome, 5, 12, seed=1):
    print(read, "an Offset", toy_genome.find(read))

In [ ]:
%%ipytest

def test_random_dna_length_and_alphabet():
    s = random_dna(1000, seed=1)
    assert len(s) == 1000
    assert set(s) <= set("ACGT")
    assert len(set(s)) == 4


def test_random_dna_reproducible():
    assert random_dna(50, seed=7) == random_dna(50, seed=7)
    assert random_dna(50, seed=7) != random_dna(50, seed=8)


def test_generate_reads_are_substrings():
    genome = random_dna(500, seed=3)
    reads = generate_reads(genome, 100, 20, seed=4)
    assert len(reads) == 100
    assert all(len(r) == 20 for r in reads)
    assert all(r in genome for r in reads)


def test_generate_reads_reaches_both_ends():
    genome = "ACGTTGCA"
    reads = generate_reads(genome, 300, 3, seed=0)
    assert "ACG" in reads           # Start 0
    assert "GCA" in reads           # letzter möglicher Start


def test_generate_reads_full_length():
    genome = random_dna(30, seed=2)
    assert generate_reads(genome, 3, 30, seed=5) == [genome] * 3

## 3. Exaktes Matching: der naive Algorithmus (Skriptum 5)

**Aufgabe 3.1:** Schreibe `naive(p, t)`. Sie liefert die Liste **aller** Offsets, an denen `p` in `t` vorkommt,
aufsteigend sortiert. Lege `p` an jedem Offset `i` von `0` bis `len(t) - len(p)` an und vergleiche Zeichen für
Zeichen; brich beim ersten Mismatch ab (Skriptum 5.2). Verwende **nicht** `str.find` oder `in`.

**Aufgabe 3.2:** Schreibe `find_all(p, t)` mit `str.find`. Rufe `t.find(p, start)` in einer Schleife auf und erhöhe
`start` nach jedem Treffer um 1, damit auch überlappende Vorkommen gefunden werden. `find_all` muss für alle Eingaben
dasselbe Ergebnis liefern wie `naive`.

**Aufgabe 3.3:** Schreibe
- `naive_with_counts(p, t)`, die zusätzlich zählt: Rückgabe `(occurrences, n_alignments, n_comparisons)`.
  Jeder Vergleich `t[i + j] != p[j]` zählt als ein Zeichenvergleich.
- `comparison_bounds(x, y)`, die für `x = |P|` und `y = |T|` das Tupel `(minimum, maximum)` der möglichen
  Zeichenvergleiche liefert (Skriptum 5.3).

**Aufgabe 3.4:** Schreibe `estimate_runtime_years(n_reads, read_len, genome_len, alignments_per_second)`. Sie
berechnet, wie viele Jahre (1 Jahr = 365 Tage) der naive Algorithmus für alle Reads braucht, wenn er
`alignments_per_second` Alignments pro Sekunde schafft (Skriptum 5.4).

In [ ]:
def naive(p: str, t: str) -> list[int]:
    """All offsets where p occurs in t (naive exact matching)."""
    # Zwei verschachtelte Schleifen: Alignments (i) und Zeichen (j), siehe Skriptum 5.2
    raise NotImplementedError("TODO: implement naive")


def find_all(p: str, t: str) -> list[int]:
    """All (also overlapping) offsets of p in t, using str.find."""
    # Tipp: t.find(p, start) liefert -1, wenn nichts mehr gefunden wird
    raise NotImplementedError("TODO: implement find_all")


def naive_with_counts(p: str, t: str) -> tuple[list[int], int, int]:
    """Like naive(), but also count alignments and character comparisons."""
    raise NotImplementedError("TODO: implement naive_with_counts")


def comparison_bounds(x: int, y: int) -> tuple[int, int]:
    """(minimum, maximum) number of character comparisons for |P| = x, |T| = y."""
    raise NotImplementedError("TODO: implement comparison_bounds")


def estimate_runtime_years(n_reads: int, read_len: int, genome_len: int, alignments_per_second: float) -> float:
    """Estimated runtime of naive matching for all reads in years (365 days)."""
    raise NotImplementedError("TODO: implement estimate_runtime_years")

In [ ]:
t = "There would have been a time for such a word"
print(t.find("word"))                        # Skriptum 5.1: 40
print(naive("word", t))                      # [40]
print(naive("AG", "AGCTTAGATAGC"))           # Skriptum 5.2: [0, 5, 9]
print(naive_with_counts("word", t))          # Skriptum 5.3: ([40], 41, 46)
print(comparison_bounds(4, 44))              # (41, 164)

# Hochrechnung aus Skriptum 5.4 (9 * 10^8 Reads, 100 nt, Humangenom, 7 Mio. Alignments pro Sekunde)
print(f"{estimate_runtime_years(9 * 10**8, 100, 3_055_000_000, 7e6):,.0f} Jahre")

In [ ]:
%%ipytest

WORD_TEXT = "There would have been a time for such a word"


def test_naive_basic():
    assert naive("AG", "AGCTTAGATAGC") == [0, 5, 9]
    assert naive("word", WORD_TEXT) == [40]


def test_naive_overlapping_and_edge_cases():
    assert naive("AA", "AAAA") == [0, 1, 2]
    assert naive("ACGT", "ACGT") == [0]
    assert naive("ACGTA", "ACGT") == []
    assert naive("GG", "ACGT") == []


def test_find_all_equals_naive():
    t = random_dna(2000, seed=11)
    for p in ["A", "AC", "ACG", "TTT", "GATTACA", t[100:130]]:
        assert find_all(p, t) == naive(p, t)
    assert find_all("AA", "AAAA") == [0, 1, 2]


def test_counts_word_example():
    assert naive_with_counts("word", WORD_TEXT) == ([40], 41, 46)


def test_counts_extremes():
    assert naive_with_counts("aaaa", "a" * 44)[1:] == (41, 164)
    assert naive_with_counts("abbb", "b" * 44) == ([], 41, 41)


def test_comparison_bounds():
    assert comparison_bounds(4, 44) == (41, 164)
    assert comparison_bounds(100, 5386) == (5287, 528700)


def test_counts_within_bounds():
    t = random_dna(500, seed=5)
    p = t[200:210]
    occ, n_al, n_cmp = naive_with_counts(p, t)
    lo, hi = comparison_bounds(len(p), len(t))
    assert 200 in occ
    assert n_al == lo
    assert lo <= n_cmp <= hi


def test_estimate_runtime_years():
    years = estimate_runtime_years(9 * 10**8, 100, 3_055_000_000, 7e6)
    assert years == pytest.approx(12_455, rel=1e-3)
    # 1 Mio. Reads gegen phiX bei 5 Mio. Alignments/s: 1057 s (Skriptum, Kontrollfrage 10)
    assert estimate_runtime_years(10**6, 100, 5386, 5e6) * 365 * 24 * 3600 == pytest.approx(1057.4)

## 4. Genom und Reads einlesen (Skriptum 6.2)

**Aufgabe 4.1:** Schreibe `read_genome(filename)`. Überspringe Zeilen, die mit `>` beginnen (FASTA-Kopfzeile), und
hänge alle übrigen Zeilen ohne Zeilenumbruch aneinander.

**Aufgabe 4.2:** Schreibe `read_fastq(filename)`. Eine FASTQ-Datei hat **vier Zeilen pro Read**: Name (`@...`),
Sequenz, Plus-Zeile (`+`) und Qualitäten. Rückgabe: `(sequences, qualities)` als zwei Listen gleicher Länge.

*Tipps:* `with open(filename) as f:`, `line.startswith(">")`, `line.rstrip()`. Für FASTQ in einer Schleife je
vier Mal `f.readline()` aufrufen; eine leere Sequenzzeile bedeutet Dateiende.

In [ ]:
def read_genome(filename: str) -> str:
    """Read a FASTA file and return the sequence without header lines."""
    raise NotImplementedError("TODO: implement read_genome")


def read_fastq(filename: str) -> tuple[list[str], list[str]]:
    """Read a FASTQ file and return (sequences, qualities)."""
    raise NotImplementedError("TODO: implement read_fastq")

In [ ]:
genome = read_genome(data_path("phix.fa"))
reads, quals = read_fastq(data_path("ERR266411_1.first1000.fastq"))
print(len(genome), genome[:30])               # Skriptum 6.2: 5386
print(collections.Counter(genome))
print(len(reads), len(reads[0]))              # 1000 100
print(reads[0])
print(quals[0])

In [ ]:
%%ipytest

def test_read_genome():
    g = read_genome(data_path("phix.fa"))
    assert len(g) == 5386
    assert g[:10] == "GAGTTTTATC"
    assert set(g) == set("ACGT")
    counts = collections.Counter(g)
    assert (counts["A"], counts["C"], counts["G"], counts["T"]) == (1291, 1157, 1254, 1684)


def test_read_fastq():
    seqs, quals = read_fastq(data_path("ERR266411_1.first1000.fastq"))
    assert len(seqs) == len(quals) == 1000
    assert all(len(s) == 100 for s in seqs)
    assert all(len(s) == len(q) for s, q in zip(seqs, quals))
    assert seqs[0].startswith("TAAACAAGCAGTAGTAATTCC")
    assert quals[0].startswith("B@DFEFFFGEGG")

## 5. Echte Reads gegen phiX174 (Skriptum 6.3 und 6.4)

**Aufgabe 5.1:** Schreibe `count_matched(reads, genome, prefix_len=None, both_strands=False)`. Sie zählt, wie viele
Reads **mindestens einen** exakten Treffer im Genom haben (verwende deine Funktion `naive`).
- Ist `prefix_len` gesetzt, wird nur der Präfix `r[:prefix_len]` jedes Reads gesucht.
- Ist `both_strands=True`, wird zusätzlich das reverse Komplement (des ggf. gekürzten Reads) gesucht.

**Aufgabe 5.2:** phiX174 ist ringförmig. Schreibe `circularize(genome, read_len)`, die an das Genom seine ersten
`read_len - 1` Basen anhängt. Dann findet exaktes Matching auch Reads, die über das Ende des Strings hinausreichen.

> Die Berechnungen mit 1000 Reads dauern mit dem naiven Algorithmus jeweils etwa eine Sekunde.

In [ ]:
def count_matched(reads: list[str], genome: str, prefix_len: int | None = None,
                  both_strands: bool = False) -> int:
    """Number of reads with at least one exact hit in genome."""
    raise NotImplementedError("TODO: implement count_matched")


def circularize(genome: str, read_len: int) -> str:
    """Append the first read_len - 1 bases so that reads crossing the origin can match."""
    raise NotImplementedError("TODO: implement circularize")

In [ ]:
# Kontrolle: künstliche, fehlerfreie Reads passen alle (Skriptum 6.3)
artificial = generate_reads(genome, 100, 100, seed=1)
print("künstliche Reads:", count_matched(artificial, genome), "/ 100")

# Echte Reads: die fünf Bedingungen aus Abb. 3 im Skriptum
start = time.perf_counter()
print("ganzer Read, ein Strang:          ", count_matched(reads, genome))
print("erste 30 Basen, ein Strang:       ", count_matched(reads, genome, prefix_len=30))
print("erste 30 Basen, beide Stränge:    ", count_matched(reads, genome, prefix_len=30, both_strands=True))
print("ganzer Read, beide Stränge:       ", count_matched(reads, genome, both_strands=True))
print("ganzer Read, beide Stränge, Kreis:", count_matched(reads, circularize(genome, 100), both_strands=True))
elapsed = time.perf_counter() - start

# Eigene Messung für Aufgabe 3.4: 5 Durchläufe mit beiden Strängen = 9 x 1000 Suchen über ca. 5300 Offsets
n_alignments = 9 * 1000 * (len(genome) - 100 + 1)
print(f"Laufzeit {elapsed:.1f} s, grob {n_alignments / elapsed:,.0f} Alignments pro Sekunde")

In [ ]:
%%ipytest

def test_artificial_reads_all_match():
    g = read_genome(data_path("phix.fa"))
    assert count_matched(generate_reads(g, 50, 100, seed=2), g) == 50


def test_count_matched_small_example():
    g = "ACGTTTGCAAC"
    reads = ["ACGT", "GCAA", "TTGG", "TTGCAAAT"]
    assert count_matched(reads, g) == 2
    assert count_matched(reads, g, prefix_len=3) == 4       # aus "TTGG" und "TTGCAAAT" wird "TTG"
    assert count_matched(["CAAA"], g) == 0
    assert count_matched(["TTGC"], "GCAAA", both_strands=True) == 1   # reverses Komplement GCAA


def test_circularize():
    assert circularize("ACGTAC", 3) == "ACGTACAC"
    assert len(circularize("A" * 5386, 100)) == 5386 + 99


def test_phix_experiment():
    g = read_genome(data_path("phix.fa"))
    seqs, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
    assert count_matched(seqs, g) == 7
    assert count_matched(seqs, g, prefix_len=30) == 459
    assert count_matched(seqs, g, prefix_len=30, both_strands=True) == 932
    assert count_matched(seqs, g, both_strands=True) == 449
    assert count_matched(seqs, circularize(g, 100), both_strands=True) == 853

## 6. Nachdenken und notieren

Beantworte die Fragen mit Hilfe deiner Ergebnisse aus Teil 5 und des Skriptums (Abschnitt 6.4). Schreibe deine
Antworten in die Markdown-Zelle darunter.

1. Warum passen nur 7 Reads in voller Länge, aber 459 mit ihren ersten 30 Basen?
2. Warum verdoppelt sich die Zahl fast, wenn man auch das reverse Komplement sucht?
3. Warum hilft `circularize` bei phiX174? Gibt es Genome, bei denen das nicht nötig wäre?
4. Für die restlichen 147 Reads findet exaktes Matching keine Stelle. Was müsste ein Algorithmus zusätzlich können?
5. Vergleiche deine gemessene Rate (Alignments pro Sekunde) mit der Hochrechnung aus Teil 3. Wie lange bräuchte
   dein Code für ein Humangenom mit 30-facher Coverage?

*Deine Antworten:*

1. 
2. 
3. 
4. 
5. 